# Patient-level Results and Statistical Analysis

This notebook combines repeated-run outputs from transfer learning (TL), a reference convolutional neural network (CNN), and ensemble learning (EL). It creates descriptive run-stability summaries and patient-level analyses from a fixed aggregation of repeated predictions.

**Inputs**
- a TL results workbook: 11 architectures × 100 runs;
- a reference-CNN results workbook: 100 runs;
- an EL results workbook: 8 configurations × 100 run-matched evaluations;
- a test-label file in the same order as the saved comma-separated test scores.

**Outputs**
- descriptive median (IQR) run-stability tables;
- seed-aggregated patient-level metrics with stratified bootstrap 95% confidence intervals;
- final patient-level confusion matrices;
- paired comparisons;
- calibration measures and plots for probability-producing methods;
- decision-curve data and uncertainty plots for probability-producing methods.


In [ ]:
# Cell 1: Imports and configuration
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from scipy.optimize import minimize
from scipy.stats import binomtest, norm
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# ── User configuration ──────────────────────────────────────────────────────
DATA_DIR         = 'data'              # Directory containing input files
TL_RESULTS_FILE  = 'results_TL.xlsx'   # TL workbook with 11 × 100 runs
CNN_RESULTS_FILE = 'results_CNN.xlsx'  # Reference-CNN workbook with 100 runs
EL_RESULTS_FILE  = 'results_EL.xlsx'   # EL workbook with 8 × 100 run-matched evaluations
LABELS_FILE      = 'test_labels.txt'   # True test-set labels in saved-score order
OUTPUT_DIR       = 'results_publish_outputs'
NUM_RUNS         = 100

THRESHOLD = 0.50
BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20261016
CALIBRATION_BINS = 5
DCA_THRESHOLDS = np.arange(0.05, 0.96, 0.05)

# Figure-panel options. Leave a list empty to display all eligible methods.
ROC_DISPLAY_METHODS = []
CALIBRATION_DISPLAY_METHODS = []
DCA_DISPLAY_METHODS = []
# ─────────────────────────────────────────────────────────────────────────────

DATA_PATH = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
if not DATA_PATH.is_dir():
    raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_PATH.resolve()}')

TL_FILE = DATA_PATH / TL_RESULTS_FILE
CNN_FILE = DATA_PATH / CNN_RESULTS_FILE
EL_FILE = DATA_PATH / EL_RESULTS_FILE
LABEL_FILE = DATA_PATH / LABELS_FILE
missing_files = [path for path in [TL_FILE, CNN_FILE, EL_FILE, LABEL_FILE] if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Missing input files:\n' + '\n'.join(str(path) for path in missing_files))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)


print('TL results :', TL_FILE)
print('CNN results:', CNN_FILE)
print('EL results :', EL_FILE)
print('Test labels:', LABEL_FILE)
print('Output directory:', OUTPUT_DIR)


In [ ]:
# Cell 2: Reading, parsing, and validation helpers
METRIC_COLUMNS = ['ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC']
METRIC_LABELS = {
    'ACC': 'Accuracy',
    'PRE': 'Precision',
    'SEN': 'Sensitivity',
    'SPE': 'Specificity',
    'F1S': 'F1-score',
    'AUC': 'AUC',
}


def read_run_workbook(path):
    sheets = pd.read_excel(path, sheet_name=None, engine='openpyxl')
    run_sheet = 'Run_results' if 'Run_results' in sheets else next(iter(sheets))
    return sheets, sheets[run_sheet].copy()


def read_test_labels(path):
    text = path.read_text(encoding='utf-8').strip()
    tokens = text.replace(',', ' ').split()
    if not tokens:
        raise ValueError(f'{path.name} does not contain test labels.')
    labels = np.asarray([int(token) for token in tokens], dtype=int)
    if not set(np.unique(labels)).issubset({0, 1}) or len(np.unique(labels)) != 2:
        raise ValueError(f'{path.name} must contain both binary outcome classes.')
    return labels


def parse_vector(value, expected_length, field_name, allow_empty=False):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        text = ''
    else:
        text = str(value).strip().strip('[](){} ')
    if not text:
        if allow_empty:
            return None
        raise ValueError(f'{field_name} is empty.')
    separator = ',' if ',' in text else None
    result = np.array([float(item) for item in text.split(separator) if item.strip()], dtype=float)
    if result.size != expected_length:
        raise ValueError(f'{field_name} contains {result.size} values; expected {expected_length}.')
    return result


def require_columns(frame, columns, source_name):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f'{source_name} is missing columns: {missing}')


def test_manifest_from_tl(sheets):
    if 'Patient_manifest' not in sheets:
        return None
    manifest = sheets['Patient_manifest'].copy()
    require_columns(manifest, ['split', 'within_split_order', 'observed_label'], 'Patient_manifest')
    test = manifest.loc[manifest['split'].astype(str).str.lower().eq('test')].copy()
    test = test.sort_values('within_split_order', kind='stable').reset_index(drop=True)
    if test.empty:
        raise ValueError('No test patients were found in Patient_manifest.')
    labels = pd.to_numeric(test['observed_label'], errors='raise').to_numpy(dtype=int)
    if not set(np.unique(labels)).issubset({0, 1}) or len(np.unique(labels)) != 2:
        raise ValueError('The test manifest must contain both binary outcome classes.')
    return test


def normalise_tl_runs(frame, n_patients):
    require_columns(
        frame,
        ['model_name', 'seed', 'test_probabilities', 'test_binary_predictions',
         'test_accuracy', 'test_precision', 'test_sensitivity', 'test_specificity',
         'test_f1', 'test_auc'],
        'TL Run_results',
    )
    result = pd.DataFrame({
        'method': frame['model_name'].astype(str),
        'family': 'TL',
        'run_id': frame['seed'],
        'has_probability': True,
        'probability_text': frame['test_probabilities'],
        'binary_text': frame['test_binary_predictions'],
        'ACC': frame['test_accuracy'],
        'PRE': frame['test_precision'],
        'SEN': frame['test_sensitivity'],
        'SPE': frame['test_specificity'],
        'F1S': frame['test_f1'],
        'AUC': frame['test_auc'],
        'source': 'TL',
    })
    return result


def normalise_cnn_runs(frame, n_patients):
    require_columns(
        frame,
        ['seed', 'test_probabilities', 'test_binary_predictions',
         'test_accuracy', 'test_precision', 'test_sensitivity', 'test_specificity',
         'test_f1', 'test_auc'],
        'CNN Run_results',
    )
    result = pd.DataFrame({
        'method': 'Reference CNN',
        'family': 'CNN',
        'run_id': frame['seed'],
        'has_probability': True,
        'probability_text': frame['test_probabilities'],
        'binary_text': frame['test_binary_predictions'],
        'ACC': frame['test_accuracy'],
        'PRE': frame['test_precision'],
        'SEN': frame['test_sensitivity'],
        'SPE': frame['test_specificity'],
        'F1S': frame['test_f1'],
        'AUC': frame['test_auc'],
        'source': 'CNN',
    })
    return result


def normalise_el_runs(frame, n_patients):
    require_columns(
        frame,
        ['pool', 'rule', 'tag', 'test_binary_predictions',
         'test_accuracy', 'test_precision', 'test_sensitivity', 'test_specificity',
         'test_f1', 'test_auc'],
        'EL Run_results',
    )
    if 'Weighted Sum' in set(frame['rule'].astype(str)) or 'Borda Count' in set(frame['rule'].astype(str)):
        raise ValueError('EL workbook contains unsupported rules.')
    valid_rules = {'Majority Voting', 'Sum Rule', 'Max Rule', 'Median Rule'}
    if not set(frame['rule'].astype(str)).issubset(valid_rules):
        raise ValueError('EL workbook contains an unrecognised ensemble rule.')
    methods = frame['pool'].astype(str) + ' ' + frame['rule'].astype(str)
    has_probability = ~frame['rule'].astype(str).eq('Majority Voting')
    probability_text = frame['test_probabilities'] if 'test_probabilities' in frame.columns else frame['predicts']
    result = pd.DataFrame({
        'method': methods,
        'family': 'EL',
        'run_id': frame['tag'],
        'has_probability': has_probability,
        'probability_text': probability_text,
        'binary_text': frame['test_binary_predictions'],
        'ACC': frame['test_accuracy'],
        'PRE': frame['test_precision'],
        'SEN': frame['test_sensitivity'],
        'SPE': frame['test_specificity'],
        'F1S': frame['test_f1'],
        'AUC': frame['test_auc'],
        'source': 'EL',
    })
    return result


def validate_run_vectors(frame, n_patients):
    for index, row in frame.iterrows():
        parse_vector(row['binary_text'], n_patients, f"{row['method']} run {row['run_id']} binary predictions")
        if bool(row['has_probability']):
            probabilities = parse_vector(row['probability_text'], n_patients, f"{row['method']} run {row['run_id']} probabilities")
            if np.any((probabilities < 0) | (probabilities > 1)):
                raise ValueError(f"{row['method']} run {row['run_id']} contains probabilities outside [0, 1].")
        elif parse_vector(row['probability_text'], n_patients, 'unused', allow_empty=True) is not None:
            raise ValueError(f"{row['method']} should not contain probability vectors.")


def validate_run_coverage(frame, expected_runs=100):
    counts = frame.groupby('method', sort=True).size()
    bad_counts = counts[counts.ne(expected_runs)]
    if not bad_counts.empty:
        raise ValueError(f'Each method must have {expected_runs} runs. Invalid coverage:\n{bad_counts}')
    duplicates = frame.duplicated(['method', 'run_id'], keep=False)
    if duplicates.any():
        problem = frame.loc[duplicates, ['method', 'run_id']].sort_values(['method', 'run_id'])
        raise ValueError(f'Duplicate method/run identifiers found:\n{problem}')
    return counts


In [ ]:
# Cell 3: Load all raw repeated-run outputs and validate the shared test-patient order
tl_sheets, tl_raw = read_run_workbook(TL_FILE)
cnn_sheets, cnn_raw = read_run_workbook(CNN_FILE)
el_sheets, el_raw = read_run_workbook(EL_FILE)

y_true = read_test_labels(LABEL_FILE)
N_PATIENTS = len(y_true)
test_manifest = test_manifest_from_tl(tl_sheets)

if test_manifest is not None:
    if len(test_manifest) != N_PATIENTS or not np.array_equal(y_true, test_manifest['observed_label'].to_numpy(dtype=int)):
        raise ValueError('TL Patient_manifest does not match LABELS_FILE in count or order.')

cnn_manifest = test_manifest_from_tl(cnn_sheets)
if cnn_manifest is not None:
    if len(cnn_manifest) != N_PATIENTS or not np.array_equal(y_true, cnn_manifest['observed_label'].to_numpy(dtype=int)):
        raise ValueError('CNN Patient_manifest does not match LABELS_FILE in count or order.')
    if test_manifest is not None and 'file_name' in test_manifest and 'file_name' in cnn_manifest:
        if not np.array_equal(test_manifest['file_name'].astype(str), cnn_manifest['file_name'].astype(str)):
            raise ValueError('CNN Patient_manifest patient order does not match the TL Patient_manifest.')

runs = pd.concat([
    normalise_tl_runs(tl_raw, N_PATIENTS),
    normalise_cnn_runs(cnn_raw, N_PATIENTS),
    normalise_el_runs(el_raw, N_PATIENTS),
], ignore_index=True)

for metric in METRIC_COLUMNS:
    runs[metric] = pd.to_numeric(runs[metric], errors='coerce')
runs['run_id'] = pd.to_numeric(runs['run_id'], errors='raise').astype(int)
runs['has_probability'] = runs['has_probability'].astype(bool)

coverage = validate_run_coverage(runs, expected_runs=NUM_RUNS)
validate_run_vectors(runs, N_PATIENTS)

method_index = (
    runs[['method', 'family', 'has_probability']]
    .drop_duplicates()
    .sort_values(['family', 'method'])
    .reset_index(drop=True)
)

print(f'Locked test cohort: {N_PATIENTS} patients ({y_true.sum()} positive; {(1-y_true).sum()} negative).')
print(f'Validated raw runs: {len(runs)} ({NUM_RUNS} per method).')
print('Methods by family:')
print(method_index.groupby('family').size().to_string())
print('\nValidated 100-run coverage:')
print(coverage.to_string())
print('\nMethod labels for optional paired comparisons:')
display(method_index)


In [ ]:
# Cell 4: Descriptive run-stability results
stability_rows = []
for (family, method), group in runs.groupby(['family', 'method'], sort=True):
    for metric in METRIC_COLUMNS:
        values = group[metric].dropna().to_numpy(dtype=float)
        if values.size == 0:
            continue
        q1, q3 = np.percentile(values, [25, 75])
        stability_rows.append({
            'family': family,
            'method': method,
            'metric': METRIC_LABELS[metric],
            'median': np.median(values),
            'q1': q1,
            'q3': q3,
            'iqr': q3 - q1,
            'minimum': values.min(),
            'maximum': values.max(),
            'n_runs': values.size,
        })
stability_long = pd.DataFrame(stability_rows)
stability_display = stability_long.pivot(index=['family', 'method'], columns='metric', values='median').reset_index()
stability_display = stability_display.reindex(columns=['family', 'method', 'Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F1-score', 'AUC'])

print('Median run-level performance across 100 repeated runs (descriptive only):')
display(stability_display.round(4))


In [ ]:
# Cell 5: Seed aggregation and patient-level bootstrap confidence intervals

def classification_metrics(y, binary_calls, probabilities=None):
    tn, fp, fn, tp = confusion_matrix(y, binary_calls, labels=[0, 1]).ravel()
    output = {
        'ACC': accuracy_score(y, binary_calls),
        'PRE': precision_score(y, binary_calls, zero_division=0),
        'SEN': recall_score(y, binary_calls, zero_division=0),
        'SPE': tn / (tn + fp) if (tn + fp) else np.nan,
        'F1S': f1_score(y, binary_calls, zero_division=0),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }
    if probabilities is not None:
        output['AUC'] = roc_auc_score(y, probabilities)
        output['Brier'] = brier_score_loss(y, probabilities)
    else:
        output['AUC'] = np.nan
        output['Brier'] = np.nan
    return output


def stratified_bootstrap_indices(y, replicates, seed):
    rng = np.random.default_rng(seed)
    class_zero = np.flatnonzero(y == 0)
    class_one = np.flatnonzero(y == 1)
    indices = []
    for _ in range(replicates):
        draw_zero = rng.choice(class_zero, size=len(class_zero), replace=True)
        draw_one = rng.choice(class_one, size=len(class_one), replace=True)
        indices.append(np.concatenate([draw_zero, draw_one]))
    return np.asarray(indices, dtype=int)


def aggregate_method(group):
    group = group.sort_values('run_id', kind='stable').reset_index(drop=True)
    binary_matrix = np.vstack([
        parse_vector(value, N_PATIENTS, f"{group.loc[i, 'method']} run {group.loc[i, 'run_id']} binary predictions")
        for i, value in enumerate(group['binary_text'])
    ]).astype(int)
    if group['has_probability'].iloc[0]:
        probability_matrix = np.vstack([
            parse_vector(value, N_PATIENTS, f"{group.loc[i, 'method']} run {group.loc[i, 'run_id']} probabilities")
            for i, value in enumerate(group['probability_text'])
        ])
        aggregate_score = probability_matrix.mean(axis=0)
        aggregate_calls = (aggregate_score >= THRESHOLD).astype(int)
        score_kind = 'mean probability'
        return aggregate_score, aggregate_calls, score_kind
    aggregate_score = binary_matrix.mean(axis=0)
    aggregate_calls = (aggregate_score > 0.5).astype(int)
    score_kind = 'positive vote fraction'
    return aggregate_score, aggregate_calls, score_kind


def bootstrap_metric_values(y, calls, probabilities, indices):
    """Vectorised stratified patient-bootstrap metric values."""
    sampled_y = y[indices]
    sampled_calls = calls[indices]
    positive = sampled_y == 1
    negative = ~positive
    tp = np.sum((sampled_calls == 1) & positive, axis=1)
    tn = np.sum((sampled_calls == 0) & negative, axis=1)
    fp = np.sum((sampled_calls == 1) & negative, axis=1)
    fn = np.sum((sampled_calls == 0) & positive, axis=1)
    denominator = tp + fp
    precision = np.divide(tp, denominator, out=np.zeros_like(tp, dtype=float), where=denominator != 0)
    recall = np.divide(tp, tp + fn, out=np.full_like(tp, np.nan, dtype=float), where=(tp + fn) != 0)
    specificity = np.divide(tn, tn + fp, out=np.full_like(tn, np.nan, dtype=float), where=(tn + fp) != 0)
    f1_denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, f1_denominator, out=np.zeros_like(tp, dtype=float), where=f1_denominator != 0)
    result = {
        'ACC': (tp + tn) / sampled_y.shape[1],
        'PRE': precision,
        'SEN': recall,
        'SPE': specificity,
        'F1S': f1,
    }
    if probabilities is not None:
        sampled_probabilities = probabilities[indices]
        n_negative = int(np.sum(y == 0))
        negative_scores = sampled_probabilities[:, :n_negative]
        positive_scores = sampled_probabilities[:, n_negative:]
        pairwise_difference = positive_scores[:, :, None] - negative_scores[:, None, :]
        result['AUC'] = (
            (pairwise_difference > 0).mean(axis=(1, 2))
            + 0.5 * (pairwise_difference == 0).mean(axis=(1, 2))
        )
        result['Brier'] = np.mean((sampled_probabilities - sampled_y) ** 2, axis=1)
    return result


def bootstrap_metric_intervals(y, calls, probabilities, indices):
    samples = bootstrap_metric_values(y, calls, probabilities, indices)
    return {
        name: (np.percentile(values, 2.5), np.percentile(values, 97.5))
        for name, values in samples.items()
    }

bootstrap_indices = stratified_bootstrap_indices(y_true, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED)

patient_prediction_rows = []
patient_summary_rows = []
confusion_rows = []
method_outputs = {}

method_groups = list(runs.groupby(['family', 'method'], sort=True))
for method_number, ((family, method), group) in enumerate(method_groups, start=1):
    print(f'Calculating patient-bootstrap confidence intervals: {method_number}/{len(method_groups)} — {method}')
    aggregate_score, aggregate_calls, score_kind = aggregate_method(group)
    probabilities = aggregate_score if group['has_probability'].iloc[0] else None
    values = classification_metrics(y_true, aggregate_calls, probabilities)
    intervals = bootstrap_metric_intervals(y_true, aggregate_calls, probabilities, bootstrap_indices)
    method_outputs[method] = {
        'family': family,
        'calls': aggregate_calls,
        'probabilities': probabilities,
        'aggregate_score': aggregate_score,
        'score_kind': score_kind,
        'metrics': values,
    }
    for patient_index, (score, call) in enumerate(zip(aggregate_score, aggregate_calls), start=1):
        patient_prediction_rows.append({
            'method': method,
            'family': family,
            'within_test_order': patient_index,
            'true_label': int(y_true[patient_index - 1]),
            'aggregate_score': float(score),
            'score_type': score_kind,
            'final_binary_call': int(call),
        })
    for metric in ['ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC', 'Brier']:
        estimate = values.get(metric, np.nan)
        low, high = intervals.get(metric, (np.nan, np.nan))
        patient_summary_rows.append({
            'family': family,
            'method': method,
            'metric': METRIC_LABELS.get(metric, metric),
            'estimate': estimate,
            'ci_95_low': low,
            'ci_95_high': high,
            'n_patients': N_PATIENTS,
            'n_positive': int(y_true.sum()),
            'n_negative': int((1 - y_true).sum()),
        })
    confusion_rows.append({
        'family': family,
        'method': method,
        'TP': values['TP'],
        'TN': values['TN'],
        'FP': values['FP'],
        'FN': values['FN'],
    })

patient_predictions = pd.DataFrame(patient_prediction_rows)
patient_summary_long = pd.DataFrame(patient_summary_rows)
patient_summary_wide = patient_summary_long.pivot(index=['family', 'method'], columns='metric', values='estimate').reset_index()
patient_summary_wide = patient_summary_wide.reindex(columns=['family', 'method', 'Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F1-score', 'AUC', 'Brier'])
patient_confusion = pd.DataFrame(confusion_rows).sort_values(['family', 'method']).reset_index(drop=True)

print('Patient-level performance after repeated-run aggregation:')
display(patient_summary_wide.round(4))
print('Patient-level confusion matrices:')
display(patient_confusion)


In [ ]:
# Cell 6: Patient-level paired comparisons

def midrank(values):
    values = np.asarray(values)
    order = np.argsort(values)
    sorted_values = values[order]
    ranks = np.zeros(len(values), dtype=float)
    start = 0
    while start < len(values):
        end = start
        while end + 1 < len(values) and sorted_values[end + 1] == sorted_values[start]:
            end += 1
        ranks[start:end + 1] = 0.5 * (start + end) + 1
        start = end + 1
    result = np.empty(len(values), dtype=float)
    result[order] = ranks
    return result


def fast_delong(predictions_sorted, n_positive):
    n_negative = predictions_sorted.shape[1] - n_positive
    positive = predictions_sorted[:, :n_positive]
    negative = predictions_sorted[:, n_positive:]
    n_methods = predictions_sorted.shape[0]
    tx = np.empty((n_methods, n_positive), dtype=float)
    ty = np.empty((n_methods, n_negative), dtype=float)
    tz = np.empty((n_methods, n_positive + n_negative), dtype=float)
    for row in range(n_methods):
        tx[row] = midrank(positive[row])
        ty[row] = midrank(negative[row])
        tz[row] = midrank(predictions_sorted[row])
    aucs = tz[:, :n_positive].sum(axis=1) / n_positive / n_negative - (n_positive + 1.0) / (2.0 * n_negative)
    v01 = (tz[:, :n_positive] - tx) / n_negative
    v10 = 1.0 - (tz[:, n_positive:] - ty) / n_positive
    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / n_positive + sy / n_negative


def paired_delong(y, first, second):
    order = np.argsort(-y)
    n_positive = int(y.sum())
    predictions = np.vstack([first, second])[:, order]
    aucs, covariance = fast_delong(predictions, n_positive)
    variance = covariance[0, 0] + covariance[1, 1] - 2 * covariance[0, 1]
    if not np.isfinite(variance) or variance <= 0:
        return aucs[0], aucs[1], np.nan
    z_score = (aucs[0] - aucs[1]) / np.sqrt(variance)
    return aucs[0], aucs[1], 2 * norm.sf(abs(z_score))


def paired_difference_intervals(y, first, second, indices):
    first_metrics = bootstrap_metric_values(y, first['calls'], first['probabilities'], indices)
    second_metrics = bootstrap_metric_values(y, second['calls'], second['probabilities'], indices)
    result = {}
    for metric in ['ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC']:
        if metric not in first_metrics or metric not in second_metrics:
            continue
        differences = first_metrics[metric] - second_metrics[metric]
        result[metric] = (np.nanpercentile(differences, 2.5), np.nanpercentile(differences, 97.5))
    return result


def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        return adjusted
    valid_positions = np.flatnonzero(valid)
    ordered_positions = valid_positions[np.argsort(values[valid])]
    ordered_values = values[ordered_positions]
    n_tests = len(ordered_values)
    running_maximum = 0.0
    for rank, (position, value) in enumerate(zip(ordered_positions, ordered_values)):
        running_maximum = max(running_maximum, (n_tests - rank) * value)
        adjusted[position] = min(running_maximum, 1.0)
    return adjusted


reference_method = 'Reference CNN'
comparison_sets = {
    'TL versus reference CNN': [
        method for method, item in method_outputs.items()
        if item['family'] == 'TL'
    ],
    'EL versus reference CNN': [
        method for method, item in method_outputs.items()
        if item['family'] == 'EL'
    ],
}

paired_rows = []
for comparison_set, compared_methods in comparison_sets.items():
    for method in compared_methods:
        first = method_outputs[method]
        second = method_outputs[reference_method]
        intervals = paired_difference_intervals(y_true, first, second, bootstrap_indices)
        for metric in ['ACC', 'PRE', 'SEN', 'SPE', 'F1S']:
            first_value = first['metrics'][metric]
            second_value = second['metrics'][metric]
            low, high = intervals[metric]
            paired_rows.append({
                'comparison_set': comparison_set,
                'method_1': method,
                'method_2': reference_method,
                'metric': METRIC_LABELS[metric],
                'difference_method_1_minus_method_2': first_value - second_value,
                'ci_95_low': low,
                'ci_95_high': high,
                'test': 'Bootstrap difference CI',
                'test_family': '',
                'p_value': np.nan,
            })
        if first['probabilities'] is not None and second['probabilities'] is not None:
            auc_first, auc_second, delong_p = paired_delong(y_true, first['probabilities'], second['probabilities'])
            low, high = intervals['AUC']
            paired_rows.append({
                'comparison_set': comparison_set,
                'method_1': method,
                'method_2': reference_method,
                'metric': 'AUC',
                'difference_method_1_minus_method_2': auc_first - auc_second,
                'ci_95_low': low,
                'ci_95_high': high,
                'test': 'Paired DeLong',
                'test_family': 'Paired DeLong',
                'p_value': delong_p,
            })
        b = int(np.sum((first['calls'] == 1) & (second['calls'] == 0)))
        c = int(np.sum((first['calls'] == 0) & (second['calls'] == 1)))
        mcnemar_p = binomtest(min(b, c), n=b + c, p=0.5, alternative='two-sided').pvalue if (b + c) else 1.0
        paired_rows.append({
            'comparison_set': comparison_set,
            'method_1': method,
            'method_2': reference_method,
            'metric': 'Binary classification',
            'difference_method_1_minus_method_2': np.nan,
            'ci_95_low': np.nan,
            'ci_95_high': np.nan,
            'test': f'Exact McNemar (discordant: {b} vs {c})',
            'test_family': 'Exact McNemar',
            'p_value': mcnemar_p,
        })

paired_comparisons = pd.DataFrame(paired_rows)
paired_comparisons['p_value_holm'] = np.nan
for (comparison_set, test_family), index in paired_comparisons.groupby(['comparison_set', 'test_family']).groups.items():
    if not test_family:
        continue
    positions = list(index)
    paired_comparisons.loc[positions, 'p_value_holm'] = holm_adjust(
        paired_comparisons.loc[positions, 'p_value'].to_numpy(dtype=float)
    )

comparison_summary = paired_comparisons.loc[
    paired_comparisons['test'].eq('Paired DeLong') | paired_comparisons['test'].astype(str).str.startswith('Exact McNemar'),
    ['comparison_set', 'method_1', 'metric', 'difference_method_1_minus_method_2', 'ci_95_low', 'ci_95_high', 'test', 'p_value', 'p_value_holm']
].reset_index(drop=True)

print('Paired patient-level comparisons against the reference CNN:')
display(comparison_summary.round(4))


In [ ]:
# Cell 7: Calibration measures and calibration curves

def logit(values, epsilon=1e-6):
    clipped = np.clip(np.asarray(values, dtype=float), epsilon, 1 - epsilon)
    return np.log(clipped / (1 - clipped))


def calibration_intercept_slope(y, probabilities):
    x = logit(probabilities)
    if np.allclose(x, x[0]):
        return np.nan, np.nan

    def objective(parameters):
        linear = parameters[0] + parameters[1] * x
        return np.sum(np.logaddexp(0, linear) - y * linear)

    result = minimize(objective, x0=np.array([0.0, 1.0]), method='BFGS')
    if not result.success or not np.all(np.isfinite(result.x)):
        return np.nan, np.nan
    return float(result.x[0]), float(result.x[1])


def equal_frequency_calibration_bins(y, probabilities, n_bins):
    frame = pd.DataFrame({'observed': y, 'predicted': probabilities}).sort_values('predicted', kind='stable').reset_index(drop=True)
    groups = np.array_split(frame.index.to_numpy(), min(n_bins, len(frame)))
    return pd.DataFrame([
        {
            'bin': index + 1,
            'n_patients': len(group),
            'mean_predicted_probability': frame.loc[group, 'predicted'].mean(),
            'observed_frequency': frame.loc[group, 'observed'].mean(),
        }
        for index, group in enumerate(groups) if len(group)
    ])

calibration_rows = []
calibration_curve_rows = []
for method, item in method_outputs.items():
    if item['probabilities'] is None:
        continue
    intercept, slope = calibration_intercept_slope(y_true, item['probabilities'])
    calibration_rows.append({
        'family': item['family'],
        'method': method,
        'Brier_score': brier_score_loss(y_true, item['probabilities']),
        'calibration_intercept': intercept,
        'calibration_slope': slope,
    })
    bins = equal_frequency_calibration_bins(y_true, item['probabilities'], CALIBRATION_BINS)
    bins.insert(0, 'method', method)
    bins.insert(0, 'family', item['family'])
    calibration_curve_rows.append(bins)

calibration_summary = pd.DataFrame(calibration_rows).sort_values(['family', 'method']).reset_index(drop=True)
calibration_curves = pd.concat(calibration_curve_rows, ignore_index=True)

print('Calibration measures for probability-producing methods:')
display(calibration_summary.round(4))


In [ ]:
# Cell 8: Decision-curve analysis with patient-bootstrap uncertainty

def net_benefit(y, probabilities, threshold):
    calls = probabilities >= threshold
    tp = np.sum((calls == 1) & (y == 1))
    fp = np.sum((calls == 1) & (y == 0))
    return tp / len(y) - fp / len(y) * threshold / (1 - threshold)


def dca_with_intervals(y, probabilities, thresholds, indices):
    sampled_y = y[indices]
    sampled_probabilities = probabilities[indices]
    rows = []
    for threshold in thresholds:
        point = net_benefit(y, probabilities, threshold)
        sampled_calls = sampled_probabilities >= threshold
        tp = np.sum((sampled_calls == 1) & (sampled_y == 1), axis=1)
        fp = np.sum((sampled_calls == 1) & (sampled_y == 0), axis=1)
        boot = tp / sampled_y.shape[1] - fp / sampled_y.shape[1] * threshold / (1 - threshold)
        prevalence = y.mean()
        treat_all = prevalence - (1 - prevalence) * threshold / (1 - threshold)
        rows.append({
            'threshold_probability': threshold,
            'net_benefit': point,
            'ci_95_low': np.percentile(boot, 2.5),
            'ci_95_high': np.percentile(boot, 97.5),
            'treat_all': treat_all,
            'treat_none': 0.0,
        })
    return pd.DataFrame(rows)

dca_frames = []
dca_method_items = [(method, item) for method, item in method_outputs.items() if item['probabilities'] is not None]
for method_number, (method, item) in enumerate(dca_method_items, start=1):
    print(f'Calculating decision-curve uncertainty: {method_number}/{len(dca_method_items)} — {method}')
    frame = dca_with_intervals(y_true, item['probabilities'], DCA_THRESHOLDS, bootstrap_indices)
    frame.insert(0, 'method', method)
    frame.insert(0, 'family', item['family'])
    dca_frames.append(frame)
dca_results = pd.concat(dca_frames, ignore_index=True)

print(f'Decision-curve estimates calculated for {dca_results["method"].nunique()} probability-producing methods.')
display(dca_results.head(12).round(4))


In [ ]:
# Cell 9: Figures based on seed-aggregated patient-level results
# The visual settings below follow the submitted results notebooks: white
# panels, a light grid, bold axis labels, Matplotlib default colours, and
# an outside legend for multi-method figures.
plt.style.use('default')
plt.rcParams.update({
    'font.size': 12,
    'axes.facecolor': '#FFFFFF',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.color': 'grey',
    'grid.alpha': 0.45,
    'grid.linewidth': 0.8,
    'figure.dpi': 140,
    'savefig.dpi': 300,
})

LINE_STYLES = ['-', '--', '-.', ':']
MARKERS = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', 'd']
DEFAULT_COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']


def style_axis(axis, x_label, y_label):
    axis.set_xlabel(x_label, fontsize=20, fontweight='bold', labelpad=15)
    axis.set_ylabel(y_label, fontsize=20, fontweight='bold', labelpad=15)
    axis.tick_params(axis='both', labelsize=14)
    for label in list(axis.get_xticklabels()) + list(axis.get_yticklabels()):
        label.set_fontweight('bold')
    axis.grid(True, linewidth=0.8, alpha=0.7)
    for spine in axis.spines.values():
        spine.set_linewidth(0.8)


def save_figure(figure, filename):
    path = OUTPUT_DIR / filename
    figure.savefig(path, dpi=300, bbox_inches='tight')
    print('Saved:', path)


def plot_roc(methods, filename):
    figure, axis = plt.subplots(figsize=(14, 8))
    for index, method in enumerate(methods):
        item = method_outputs[method]
        if item['probabilities'] is None:
            continue
        fpr, tpr, _ = roc_curve(y_true, item['probabilities'])
        auc = item['metrics']['AUC']
        axis.plot(
            fpr, tpr,
            label=f'{method} (AUC = {auc:.2f})',
            color=DEFAULT_COLORS[index % len(DEFAULT_COLORS)],
            linestyle=LINE_STYLES[index % len(LINE_STYLES)],
            marker=MARKERS[index % len(MARKERS)],
            markevery=(index % 10, 10),
            linewidth=1.8,
            markersize=5,
            solid_capstyle='round',
        )
    axis.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Random classification')
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.05)
    style_axis(axis, 'False Positive Rate', 'True Positive Rate')
    axis.legend(
        title='Models',
        bbox_to_anchor=(1.03, 0.5),
        loc='center left',
        fontsize=9,
        title_fontsize=10,
        frameon=False,
    )
    figure.tight_layout(rect=[0, 0, 0.72, 1])
    save_figure(figure, filename)
    plt.show()


def plot_calibration_panels(methods, filename):
    n_cols = min(3, len(methods))
    n_rows = int(np.ceil(len(methods) / n_cols))
    figure, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.8 * n_rows), squeeze=False)
    for axis, method in zip(axes.ravel(), methods):
        frame = calibration_curves[calibration_curves['method'] == method]
        axis.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Perfect calibration')
        axis.plot(
            frame['mean_predicted_probability'],
            frame['observed_frequency'],
            marker='o',
            color='blue',
            linewidth=2,
            markersize=5,
            label=method,
        )
        axis.set_title(method, fontsize=14, fontweight='bold', pad=10)
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1.05)
        style_axis(axis, 'Mean Predicted Probability', 'Observed Frequency')
        axis.legend(fontsize=8, frameon=False, loc='upper left')
    for axis in axes.ravel()[len(methods):]:
        axis.axis('off')
    figure.tight_layout(pad=2.0)
    save_figure(figure, filename)
    plt.show()


def plot_dca_panels(methods, filename):
    n_cols = min(3, len(methods))
    n_rows = int(np.ceil(len(methods) / n_cols))
    figure, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.0 * n_rows), squeeze=False)
    for axis, method in zip(axes.ravel(), methods):
        frame = dca_results[dca_results['method'] == method]
        axis.plot(
            frame['threshold_probability'], frame['net_benefit'],
            color='blue', linewidth=2, label=method, solid_capstyle='round'
        )
        axis.fill_between(
            frame['threshold_probability'], frame['ci_95_low'], frame['ci_95_high'],
            color='blue', alpha=0.16, linewidth=0
        )
        axis.plot(frame['threshold_probability'], frame['treat_all'], color='black', linestyle='--', linewidth=1.2, alpha=0.55, label='Treat all')
        axis.axhline(0, color='black', linestyle='-.', linewidth=1.2, alpha=0.55, label='Treat none')
        visible = frame[frame['threshold_probability'] <= 0.50]
        lower = min(-0.05, float(visible[['ci_95_low', 'treat_all']].min().min()))
        upper = max(0.05, float(visible['ci_95_high'].max()))
        margin = 0.05 * (upper - lower)
        axis.set_title(method, fontsize=14, fontweight='bold', pad=10)
        axis.set_xlim(DCA_THRESHOLDS.min(), 0.50)
        axis.set_ylim(lower - margin, upper + margin)
        style_axis(axis, 'Threshold Probability', 'Net Benefit')
        axis.legend(fontsize=8, frameon=False, loc='upper right')
    for axis in axes.ravel()[len(methods):]:
        axis.axis('off')
    figure.tight_layout(pad=2.0)
    save_figure(figure, filename)
    plt.show()


probability_methods = [method for method, item in method_outputs.items() if item['probabilities'] is not None]
tl_probability_methods = [method for method in probability_methods if method_outputs[method]['family'] == 'TL']
el_probability_methods = [method for method in probability_methods if method_outputs[method]['family'] == 'EL']

plot_roc(ROC_DISPLAY_METHODS or tl_probability_methods + ['Reference CNN'], 'F1_TL_CNN_ROC.png')
plot_roc(el_probability_methods, 'F2_EL_ROC.png')
plot_calibration_panels(CALIBRATION_DISPLAY_METHODS or probability_methods, 'F3_calibration_panels.png')
plot_dca_panels(DCA_DISPLAY_METHODS or probability_methods, 'F4_DCA_panels.png')


In [ ]:
# Cell 10: Export all tables and patient-level prediction scores
stability_wide = stability_long.pivot(index=['family', 'method'], columns='metric', values=['median', 'iqr', 'q1', 'q3', 'minimum', 'maximum']).reset_index()
stability_wide.columns = [
    first if not second else f'{first}_{second}'
    for first, second in stability_wide.columns.to_flat_index()
]

export_file = OUTPUT_DIR / 'results_publish_tables.xlsx'
with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    stability_long.to_excel(writer, sheet_name='Run_stability_long', index=False)
    stability_wide.to_excel(writer, sheet_name='Run_stability_wide', index=False)
    patient_summary_long.to_excel(writer, sheet_name='Patient_metrics_CI', index=False)
    patient_summary_wide.to_excel(writer, sheet_name='Patient_metrics_wide', index=False)
    patient_confusion.to_excel(writer, sheet_name='Patient_confusion', index=False)
    patient_predictions.to_excel(writer, sheet_name='Patient_predictions', index=False)
    paired_comparisons.to_excel(writer, sheet_name='Paired_comparisons', index=False)
    calibration_summary.to_excel(writer, sheet_name='Calibration', index=False)
    calibration_curves.to_excel(writer, sheet_name='Calibration_bins', index=False)
    dca_results.to_excel(writer, sheet_name='DCA', index=False)
    if test_manifest is not None:
        test_manifest.to_excel(writer, sheet_name='Test_manifest', index=False)

print('Saved tables:', export_file)
print('Saved figure files:')
for figure_file in sorted(OUTPUT_DIR.glob('F*.png')):
    print(' -', figure_file.name)
